# SafeChain Pipeline — Integration Testing

Test the full agentic pipeline in the private environment using SafeChain + GPT-4.1.

**Run this notebook in the deployment environment where `safechain` is installed.**

Tests are layered — run top to bottom, each section builds on the previous:
1. SafeChain connection
2. Raw LLM call
3. Adapter (tool calling loop)
4. Firewall retry stack
5. Single specialist
6. Full pipeline (team → specialists → compare → synthesize)

---
## 0. Setup

In [ ]:
import os, sys, json

# Resolve project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if os.path.basename(os.getcwd()) != 'notebooks':
    PROJECT_ROOT = os.getcwd()

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'simulated')
PROFILE_DIR = os.path.join(PROJECT_ROOT, 'config', 'data_profiles')
PILLAR_DIR = os.path.join(PROJECT_ROOT, 'config', 'pillars')

print(f'Project root: {PROJECT_ROOT}')
print(f'Data dir exists: {os.path.isdir(DATA_DIR)}')
print(f'Profile dir exists: {os.path.isdir(PROFILE_DIR)}')

In [ ]:
# Load data layer
from data.gateway import SimulatedDataGateway
from data.catalog import DataCatalog
from tools.data_tools import init_tools

gw = SimulatedDataGateway.from_case_folders(DATA_DIR)
catalog = DataCatalog(profile_dir=PROFILE_DIR)
init_tools(gw, catalog)

CASE_ID = gw.list_case_ids()[0]
gw.set_case(CASE_ID)
print(f'Case: {CASE_ID}')
print(f'Tables: {gw.list_tables()}')

---
## 1. Test SafeChain Connection

Verify that `safechain` is importable and we can create an LLM instance.

In [ ]:
# ══════════════════════════════════════════════════════════════
# SAFECHAIN SEAM — configure your model name here
# ══════════════════════════════════════════════════════════════
SAFECHAIN_MODEL = "gpt-4.1"  # Change this to your environment's model name
# ══════════════════════════════════════════════════════════════

try:
    from safechain.prompts import ValidChatPromptTemplate
    from safechain.lcel import model as safechain_model
    print('safechain imported successfully')
    
    llm = safechain_model(SAFECHAIN_MODEL)
    print(f'LLM instance created: {type(llm).__name__}')
    print(f'Model: {SAFECHAIN_MODEL}')
except ImportError as e:
    print(f'ERROR: safechain not available — {e}')
    print('This notebook must be run in the deployment environment.')
    llm = None

---
## 2. Test Raw LLM Call

Send a simple message through SafeChain to verify the LLM responds.

In [ ]:
if llm is not None:
    chain = ValidChatPromptTemplate.from_messages([
        ("human", "{__input__}"),
    ]) | llm

    response = chain.invoke({"__input__": "Context:\nYou are a helpful assistant.\n\nRequest:\nSay hello in one sentence."})
    print(f'Response: {response.content}')
    print(f'Type: {type(response).__name__}')
else:
    print('Skipped — no LLM available')

---
## 3. Test SafeChain Adapter

Verify the adapter correctly wraps SafeChain with neutral role labels and JSON output parsing.

In [ ]:
from gateway.safechain_adapter import SafeChainAdapter

adapter = SafeChainAdapter(llm=llm, model_name=SAFECHAIN_MODEL)
print(f'Adapter created: {type(adapter).__name__}')

In [ ]:
# Test simple run() — no tools, expect JSON output
result = adapter.run(
    system_prompt="You are a credit risk analyst. Respond with JSON only.",
    user_message='Analyze this: FICO score is 650, 3 delinquent trades. Respond with: {"risk_level": "...", "reasoning": "..."}',
)
print('=== Adapter run() result ===')
print(json.dumps(result, indent=2))

In [ ]:
# Test chat_turn()
response = adapter.chat_turn([
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What does a FICO score of 650 indicate? Answer in one sentence."},
])
print('=== chat_turn() result ===')
print(response)

---
## 4. Test Adapter with Tool Calling

Verify the adapter can inject tool schemas, parse tool calls, dispatch them, and return final output.

In [ ]:
from tools.data_tools import list_available_tables, get_table_schema, query_table

result = adapter.run(
    system_prompt="You are a data analyst. Use the available tools to answer the question. Respond with JSON output when done.",
    user_message='What tables are available for this case? List them, then get the schema of bureau_full. '
                 'Respond with: {"output": {"tables": [...], "bureau_schema_columns": [...]}}',
    tools=[list_available_tables, get_table_schema, query_table],
)
print('=== Tool-calling result ===')
print(json.dumps(result, indent=2))

---
## 5. Test Firewall Retry Stack

Verify the FirewallStack wraps calls correctly and logs events.

In [ ]:
from gateway.firewall_stack import FirewallStack
from log.event_logger import EventLogger

logger = EventLogger(session_id='test-safechain', log_dir=os.path.join(PROJECT_ROOT, 'logs'))
firewall = FirewallStack(adapter=adapter, logger=logger)

result = firewall.call(
    system_prompt="You are a credit analyst. Respond with JSON.",
    user_message='What risk level is a customer with FICO 620 and 5 delinquent trades? '
                 'Respond: {"risk_level": "...", "reasoning": "..."}',
)
print(f'Status: {result.status}')
print(f'Data: {json.dumps(result.data, indent=2)}')
print(f'Steps recorded: {len(firewall.step_history)}')

---
## 6. Test Single Specialist

Run one Base Specialist Agent (Bureau domain) against the simulated data.

In [ ]:
from agents.base_agent import BaseSpecialistAgent
from skills.domain.loader import load_domain_skill
from config.pillar_loader import PillarLoader

# Load domain skill and pillar config
bureau_skill = load_domain_skill('bureau')
pillar_loader = PillarLoader(pillar_dir=PILLAR_DIR)
pillar_config = pillar_loader.get_specialist_config('credit_risk', 'bureau') or {}

print(f'Domain skill: {bureau_skill.name}')
print(f'Pillar config: {pillar_config}')

In [ ]:
# Create a fresh firewall for the specialist
spec_firewall = FirewallStack(adapter=adapter, logger=logger)

# Create specialist
bureau_agent = BaseSpecialistAgent(
    domain_skill=bureau_skill,
    pillar_yaml=pillar_config,
    firewall=spec_firewall,
    logger=logger,
)

# Run a question
output = bureau_agent.run(
    question="What is the delinquency risk for this customer based on bureau data?",
    mode="chat",
)

print(f'=== Bureau Specialist Output ===')
print(f'Domain: {output.domain}')
print(f'Mode: {output.mode}')
print(f'Findings: {output.findings}')
print(f'Evidence: {output.evidence}')
print(f'Implications: {output.implications}')
print(f'Data gaps: {output.data_gaps}')

---
## 7. Test Session Registry & Specialist Reuse

In [ ]:
from agents.session_registry import SessionRegistry

registry = SessionRegistry()

# First question — creates the specialist
agent1 = registry.get_or_create(
    domain='bureau', pillar='credit_risk',
    domain_skill=bureau_skill, pillar_yaml=pillar_config,
    firewall=spec_firewall, logger=logger,
)
out1 = agent1.run("What are the FICO and SBFE scores?", mode="chat")
print(f'Q1 findings: {out1.findings[:200]}...')
print(f'Rolling summary length: {len(agent1.rolling_summary)} chars')
print()

# Second question — reuses the same specialist (rolling summary carries forward)
agent2 = registry.get_or_create(
    domain='bureau', pillar='credit_risk',
    domain_skill=bureau_skill, pillar_yaml=pillar_config,
    firewall=spec_firewall, logger=logger,
)
print(f'Same instance: {agent1 is agent2}')
out2 = agent2.run("Are there any delinquent external trades?", mode="chat")
print(f'Q2 findings: {out2.findings[:200]}...')
print(f'Rolling summary length: {len(agent2.rolling_summary)} chars')
print()

# Check active registry
print(f'Active specialists: {registry.list_active()}')

---
## 8. Test General Specialist — Compare

Run two specialists, then have the General Specialist compare their outputs.

In [ ]:
from agents.general_specialist import GeneralSpecialist

# Run a second specialist (crossbu)
crossbu_skill = load_domain_skill('crossbu')
crossbu_config = pillar_loader.get_specialist_config('credit_risk', 'crossbu') or {}
crossbu_firewall = FirewallStack(adapter=adapter, logger=logger)

crossbu_agent = BaseSpecialistAgent(
    domain_skill=crossbu_skill,
    pillar_yaml=crossbu_config,
    firewall=crossbu_firewall,
    logger=logger,
)

question = "What is the overall credit risk for this customer?"
bureau_output = bureau_agent.run(question, mode="chat")
crossbu_output = crossbu_agent.run(question, mode="chat")

print(f'Bureau findings: {bureau_output.findings[:150]}...')
print(f'CrossBU findings: {crossbu_output.findings[:150]}...')

In [ ]:
# General Specialist — Compare
compare_firewall = FirewallStack(adapter=adapter, logger=logger)
general = GeneralSpecialist(firewall=compare_firewall, logger=logger)

review_report = general.compare(
    specialist_outputs={'bureau': bureau_output, 'crossbu': crossbu_output},
    question=question,
)

print(f'=== Review Report ===')
print(f'Resolved contradictions: {len(review_report.resolved)}')
for r in review_report.resolved:
    print(f'  Pair: {r.pair}')
    print(f'  Contradiction: {r.contradiction}')
    print(f'  Conclusion: {r.conclusion}')
    print()

print(f'Open conflicts: {len(review_report.open_conflicts)}')
for c in review_report.open_conflicts:
    print(f'  Pair: {c.pair}')
    print(f'  Reason: {c.reason_unresolved}')
    print()

print(f'Cross-domain insights: {review_report.cross_domain_insights}')

---
## 9. Test Full Pipeline — Orchestrator Synthesize

In [ ]:
from orchestrator.orchestrator import Orchestrator
from orchestrator.chat_agent import ChatAgent

synth_firewall = FirewallStack(adapter=adapter, logger=logger)
orchestrator = Orchestrator(
    firewall=synth_firewall,
    logger=logger,
    registry=registry,
    pillar='credit_risk',
)

final = orchestrator.synthesize(
    specialist_outputs={'bureau': bureau_output, 'crossbu': crossbu_output},
    review_report=review_report,
    question=question,
    mode='chat',
)

print(f'=== Final Output ===')
print(f'Answer: {final.answer}')
print(f'Resolved: {len(final.resolved_contradictions)}')
print(f'Open conflicts: {len(final.open_conflicts)}')
print(f'Data gaps: {len(final.data_gaps)}')
print(f'Blocked steps: {len(final.blocked_steps)}')
print(f'Specialists consulted: {final.specialists_consulted}')

In [ ]:
# Format for reviewer
chat_agent = ChatAgent(firewall=synth_firewall, logger=logger)
formatted = chat_agent.format_for_reviewer(final)
print(formatted)

---
## 10. Inspect Logs

Check the structured event log to verify all inter-agent communications were captured.

In [ ]:
log_path = os.path.join(PROJECT_ROOT, 'logs', 'test-safechain.jsonl')

if os.path.exists(log_path):
    with open(log_path) as f:
        events = [json.loads(line) for line in f]
    
    print(f'Total events: {len(events)}')
    print()
    
    # Event type summary
    from collections import Counter
    type_counts = Counter(e['event'] for e in events)
    print('Event types:')
    for event_type, count in type_counts.most_common():
        print(f'  {event_type}: {count}')
    
    print()
    print('=== Last 5 events ===')
    for e in events[-5:]:
        print(json.dumps(e, indent=2))
        print()
else:
    print(f'No log file found at {log_path}')

---
## 11. Firewall Behavior Test

Test how the system handles firewall rejections. This sends content that might trigger the firewall.

In [ ]:
# Test with content that might trigger firewall
test_firewall = FirewallStack(adapter=adapter, logger=logger, max_retries=2)

# This should work — clean content
result = test_firewall.call(
    system_prompt="You are a credit analyst.",
    user_message='Rate the risk: customer has good payment history. Respond: {"risk": "low"}',
)
print(f'Clean content — Status: {result.status}')
print(f'Data: {result.data}')
print()

# Check if any firewall events were logged
if os.path.exists(log_path):
    with open(log_path) as f:
        events = [json.loads(line) for line in f]
    fw_events = [e for e in events if 'firewall' in e.get('event', '')]
    if fw_events:
        print(f'Firewall events logged: {len(fw_events)}')
        for e in fw_events:
            print(f'  {e["event"]}: {e}')
    else:
        print('No firewall events — all calls passed cleanly')

---
## Summary

If all cells ran without errors:
- SafeChain connection works
- Adapter correctly formats messages with neutral role labels
- Tool calling loop works (prompt-injected schema → JSON tool_call → dispatch → continue)
- Firewall stack wraps calls and logs events
- Base Specialist Agent runs the 3-step chain (data_request → synthesize → answer)
- Session registry reuses specialists with rolling summary
- General Specialist Compare detects contradictions across domains
- Orchestrator synthesizes final output
- All events logged as structured JSON lines